In [ ]:
# Import libraries

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
plt.rcParams["figure.figsize"] = (12, 6)
plt.style.use("default")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

In [ ]:
# Import cleaned data

filepath = "../data/spike_waveforms_clean.csv"
df = pd.read_csv(
    filepath,
    header=None
)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Define time axis

sampling_rate_hz = 30000
time_ms = np.arange(df.shape[1]) / sampling_rate_hz * 1000
print("Neuron responses:", df.shape[0])
print("Time samples:", df.shape[1])
print("Duration (ms):", time_ms[-1])

In [ ]:
# Row-wise waveform features

baseline_window = 5
baseline = df.iloc[:, :baseline_window].mean(axis=1)

min_value = df.min(axis=1)
max_value = df.max(axis=1)

min_sample = df.idxmin(axis=1).astype(int)
max_sample = df.idxmax(axis=1).astype(int)

min_time_ms = time_ms[min_sample]
max_time_ms = time_ms[max_sample]

peak_to_peak = max_value - min_value

mean_value = df.mean(axis=1)
std_value = df.std(axis=1)

negative_peak_abs = min_value.abs()
positive_peak_abs = max_value.abs()

dominant_polarity = np.where(
    positive_peak_abs > negative_peak_abs,
    "positive",
    "negative"
)

biphasic = (
    (min_value < 0) &
    (max_value > 0)
)

features_df = pd.DataFrame({
    "baseline_mV": baseline,
    "min_mV": min_value,
    "max_mV": max_value,
    "peak_to_peak_mV": peak_to_peak,
    "mean_mV": mean_value,
    "std_mV": std_value,
    "time_to_min_ms": min_time_ms,
    "time_to_max_ms": max_time_ms,
    "dominant_polarity": dominant_polarity,
    "biphasic": biphasic
})

display(features_df.head())

In [ ]:
# Feature summary

summary_table = features_df.describe().round(4)
display(summary_table)

In [ ]:
# Heatmap sorting options

df_sorted_by_time_to_min = df.loc[
    features_df.sort_values("time_to_min_ms").index
]

df_sorted_by_amplitude = df.loc[
    features_df.sort_values(
        "peak_to_peak_mV",
        ascending=False
    ).index
]
df_sorted_by_polarity_and_time = df.loc[
    features_df.sort_values(
        ["dominant_polarity", "time_to_min_ms"]
    ).index
]

In [ ]:
heatmap_df = df_sorted_by_time_to_min
heatmap_df = df_sorted_by_amplitude
heatmap = df_sorted_by_polarity_and_time
heatmap = df

In [ ]:
# Generate plots and PDF

os.makedirs(
    ",,/outputs",
    exist_ok=True
)
output_pdf = "../outputs/initial_spike_waveform_plots_with_features.pdf"
with PdfPages(output_pdf) as pdf:

    # Page 1: feature summary table
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.axis("off")
    table = ax.table(
        cellText=summary_table.values,
        rowLabels=summary_table.index,
        colLabels=summary_table.columns,
        cellLoc="center",
        rowLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1.1, 1.5)
    ax.set_title(
        "Summary Statistics of Row-wise Neuron Response Features",
        fontsize=14,
        pad=20
    )
    pdf.savefig(fig, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    # Page 2: sampled neuron responses
    fig = plt.figure(figsize=(14, 8))
    sample_rows = np.random.choice(
        df.index,
        200, 
        replace=False
    )
    for row in sample_rows:
        plt.plot(
            time_ms,
            df.iloc[row, :],
            alpha=0.08,
            linewidth=1.5
        )
        plt.xlabel("Time (ms)")
        plt.ylabel("Voltage (mV)")
        plt.title("Sampled Neuron Responses")
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        # Page 3: mean response ± SD
        mean_response = df.mean(axis=0)
        std_response = df.std(axis=0)
        fig = plt.figure(figsize=(12, 6))
        plt.plot(
            time_ms,
            mean_response,
            linewidth=2,
            label="Mean response"
        )
        plt.fill_between(
            time_ms,
            mean_response - std_response,
            mean_response + std_response,
            alpha = 0.25,
            label="±1 SD"
        )
        plt.xlabel("Time (ms)")
        plt.ylabel("Voltage (mV)")
        plt.title("Mean Neuron Repsonse ± SD")
        plt.legend()
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        # Page 4: sorted heatmap
        fig = plt.figure(figsize=(20, 12), dpi=200)
        sns.heatmap(
            heatmap_df,
            cmap="seismic",
            robust=True,
            cbar=True
        )
        plt.xlabel("Time sample")
        plt.ylabel("Neuron response")
        plt.title("Neuron Response Heatmap Sprted by Time to Minimum")
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        # Page 5: voltage distribution
        fig=plt.figure(figsize=(10, 6))
        plt.hist(
            df.values.flatten(),
            bins=100
        )
        plt.xlabel("Voltage (mV)")
        plt.ylabel("Count")
        plt.title("Distribution of Voltage Values")
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.show()
        plt.close(fig)

    print("Saved PDF report:")
    print(output_pdf)